# Reproducing Kim et al. (2023) from dandiset 000631

**Paper.** Kim V, Semenov I, Kiester AS, Keppler MA, Ibey BL, Bixler JN, Colunga Biancatelli RML,
Pakhomov AG (2023). Control of the Electroporation Efficiency of Nanosecond Pulses by Swinging the
Electric Field Vector Direction. *Int J Mol Sci* 24, 10921.
[doi:10.3390/ijms241310921](https://doi.org/10.3390/ijms241310921). The claims file was written
from the version of record as served by Europe PMC (PMC10341945), with no supplement (the paper
has none) and no figures; the paper links no code.

**Dandiset.** [DANDI:000631](https://dandiarchive.org/dandiset/000631/0.231012.1508), version
0.231012.1508. The paper cites no version of the dandiset (its data statement says the data are
available on request), so the most recent published version is pinned; it is the only one. The
draft holds the same 15 assets (same asset ids and sizes).

**What the dandiset holds.** Fifteen stitched images (MemBrite and YO-PRO-1) of BPAE monolayers
after paired-pulse and single-pulse trains in the triangular array at 1 Hz and 833 kHz: the
experiment of Figure 2, whose 833 kHz dishes are also those of Figure 4. One 1 Hz control image is
deposited twice. The square-array experiment of Figure 3 and the FluoVolt strobe imaging of
Figure 5 are not in it. The files hold no pixel size, electrode positions, ROIs, angles or
measured values: all of these are computed here from the images (see the helper `swing.py`).

**Questions answered.** A reader answered one question on the claims file
(`questions/000631.yaml`): reading (a) of the background, the ROI intensity with no correction
for spontaneous uptake, is not a reading the paper allows, as Methods 4.4 says the footprints were
measured "to account for" it. The claims file now decides every claim under reading (b), the ROI
intensity minus the footprint intensity of the same dish, and keeps (a) as a variant that decides
no outcome. The claims of Figures 2 and 4 are re-run under the amended criteria; (a) is still
printed beside (b).

**Outside inputs.** None.

| Claim | Figure | Tested here |
|---|---|---|
| fig2-180-cancellation | 2 | tested |
| fig2-crossover-angle | 2 | tested |
| fig2-facilitation-plateau | 2 | tested |
| fig2-frequency-dependence | 2 | tested |
| fig2-60-fold-co-counter | 2 | tested |
| fig3-1hz-co-equals-cross | 3 | blocked: square-array data not in the dandiset (data_missing) |
| fig3-770khz-co-enhanced | 3 | blocked: data_missing |
| fig3-770khz-cross-same-as-1hz | 3 | blocked: data_missing |
| fig3-no-area-dependence | 3 | blocked: depends on fig3-1hz-co-equals-cross (data_missing) |
| fig4-unipolar-pattern | 4 | tested |
| fig5-base-sawtooth | 5 | blocked: FluoVolt strobe data not in the dandiset (data_missing) |
| fig5-apex-larger-polarization | 5 | blocked: data_missing |

The notebook streams every file (about 18 GB) into memory, one at a time, and reduces each image
to 10 x 10 pixel block means as it loads; it runs in about four minutes.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import swing

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = swing.list_assets()
dishes = [swing.load(a) for a in assets]
print(f"loaded {len(dishes)} files of dandiset {swing.DANDISET_ID} version {swing.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s; images reduced to {swing.BLOCK} x {swing.BLOCK} pixel block means")

## Data check

What the files hold, checked against the descriptions. Each check is stated under its output.

In [ ]:
info = pd.DataFrame([{**{k: d["meta"][k] for k in ("subject", "freq", "train", "date", "session_id",
                                                    "session_start_time", "shape", "dtype", "full_min", "full_max")},
                      "digest": d["meta"]["chunk_digest"][:10]} for d in dishes])
print(info.to_string(index=False))
print("\nFiles by frequency and train:")
print(pd.crosstab(info.freq, info.train).to_string())
m0 = dishes[0]["meta"]
print("\nContainers in every file (acquisition / processing / analysis / stimulus / intervals):",
      sorted({(tuple(d["meta"]["acquisition"]), tuple(d["meta"]["processing"]), tuple(d["meta"]["analysis"]),
               tuple(d["meta"]["stimulus"]), tuple(d["meta"]["intervals"])) for d in dishes}))
print("Data attributes:", m0["data_attrs"], "; chunks", m0["chunks"])
print("ImageSeries descriptions identical in all files:", len({d["meta"]["series_description"] for d in dishes}) == 1)
print("ImageSeries description:", m0["series_description"])
print("\nSession descriptions, by file group:")
for desc, grp in info.assign(desc=[d["meta"]["session_description"] for d in dishes]).groupby("desc"):
    print(f"- {len(grp)} file(s): {', '.join(grp.subject)}\n  {desc}\n")

Each file holds one stitched image of two channels (MemBrite and YO-PRO-1, in the order the
ImageSeries description gives them) and nothing else: no pixel size (the `resolution` is -1 and the
unit "n.a."), no electrode positions, no ROIs, no angles and no measured values. The conditions
are in the subject names: 1 Hz at 5 kV and 833 kHz ("0,8MHz") at 4 kV, paired ("BP") or single
("UP") trains. One 833 kHz paired file (BPAE-13 of 8-24-21) carries the description of another
experiment (a bipolar pulse between two parallel electrodes 3 mm apart, session id
"BPAE_5xBipolar_0,83MHz"); its image is checked below. The ImageSeries description says 144
images in a 12 x 12 grid "from a six-well plate" and also the 216 images of the paper; the
dishes are 35 mm dishes (Methods 4.1).

In [ ]:
# Recordings deposited more than once: compare the compressed chunks and the decoded images.
groups = info.groupby("digest").subject.apply(list)
dups = [g for g in groups if len(g) > 1]
print("Files whose image chunks are all identical:", dups if dups else "none")
for g in dups:
    a, b = [dishes[info.index[info.subject == s][0]]["img"] for s in g]
    print(f"  {g}: max abs difference of the block-mean images = {float(np.abs(a - b).max())}, "
          f"session_start_time {set(info.session_start_time[info.subject.isin(g)])}")
# Any other pair with near-identical images?
near = []
for i in range(len(dishes)):
    for j in range(i + 1, len(dishes)):
        if dishes[i]["img"].shape == dishes[j]["img"].shape and info.digest[i] != info.digest[j]:
            n0 = dishes[i]["img"].shape[1]
            a_, b_ = (dishes[k]["img"][1][n0 // 4:3 * n0 // 4:5, n0 // 4:3 * n0 // 4:5].ravel() for k in (i, j))
            r = np.corrcoef(a_, b_)[0, 1]
            near.append((info.subject[i], info.subject[j], r))
near = pd.DataFrame(near, columns=["a", "b", "r_YP"]).sort_values("r_YP", ascending=False)
print("\nHighest YP correlations between distinct files of the same image size (central half of the image):")
print(near.head(4).round(3).to_string(index=False))
keep = [i for i in range(len(dishes)) if not any(info.subject[i] in g[1:] for g in dups)]
print("\nDishes used (each recording once):", len(keep))
print(pd.crosstab(info.freq[keep], info.train[keep]).to_string())

BPAE-03 and BPAE-18 of 7-31-21 (1 Hz, single pulses) are the same recording deposited twice:
every compressed chunk is identical and they share a start time. BPAE-18 is left out, so each
recording is counted once. No other pair of files is alike (the highest correlation of the central
YP images of two files is printed above). That leaves 3 paired and 5 single dishes at 1 Hz, and 3 paired and
3 single at 833 kHz. The paper gives n = 3 for Figure 2; it does not say how many control dishes
there were. The 1 Hz controls come from two days (7-31-21: 3 dishes; 8-9-21, the day of the
paired dishes: 2); the 833 kHz controls (10-5-21) were imaged on a later day than the paired
dishes (8-24-21), with a larger stitched image.

In [ ]:
# Electrodes: the footprints are dark disks in the YP channel ("dark circles indicate electrode footprints").
free = [swing.find_electrodes_free(dishes[i]["img"][1]) for i in keep]
good = [g for g in free if min(g["scores"]) > 0.3]
template = swing.triangle_template(good)
geoms = {}
rows = []
mpb = swing.mm_per_block()
for i, g0 in zip(keep, free):
    g = swing.fit_template(dishes[i]["img"][1], template)
    geoms[i] = g
    dev = max(np.linalg.norm(g[k] - g0[k]) for k in ("apex", "base1", "base2")) * mpb
    side = lambda p, q: np.linalg.norm(g[p] - g[q]) * mpb
    rows.append({"subject": info.subject[i], "base_mm": side("base1", "base2"), "apex_base1_mm": side("apex", "base1"),
                 "apex_base2_mm": side("apex", "base2"),
                 "midpoint_to_apex_mm": np.linalg.norm(g["apex"] - (g["base1"] + g["base2"]) / 2) * mpb,
                 "score_apex": g["scores"][0], "score_min": min(g["scores"]),
                 "free_min_score": min(g0["scores"]), "moved_from_free_mm": dev})
geo = pd.DataFrame(rows)
print(f"Template from the {len(good)} dishes whose three strongest footprints all score above 0.3 "
      f"(base {np.linalg.norm(template[1] - template[2]) * mpb:.2f} mm, height {abs(template[0, 0] - template[1, 0]) * mpb:.2f} mm, at {swing.UM_PER_PX} um per pixel)")
print(geo.round(2).to_string(index=False))
print("\nSpread of the triangle over dishes (s.d., mm):", geo[["base_mm", "apex_base1_mm", "apex_base2_mm"]].std().round(3).to_dict())

The three electrodes are found in every dish as dark disks in the YP image: the free search (the
three strongest disks) and the fit of a rigid triangle agree to the block in all dishes but one
(BPAE-20 of 8-24-21), where the base electrodes have almost no uptake around them and the free
search takes a wrong third point; the triangle fit finds them. The triangle has the same size in
every dish, including the 833 kHz controls imaged on another day at a larger image size, so the
same array and the same image scale were used throughout. The overview below shows the fit.

In [ ]:
# Pixel size: the files give none. Checks of 0.65 um per pixel (6.5 um camera pixels, 10x objective).
ext = pd.DataFrame([{"subject": info.subject[i], "image_mm": dishes[i]["meta"]["shape"][0] * swing.UM_PER_PX / 1000} for i in keep])
print("Stitched image width at 0.65 um/pixel (mm):", sorted(set(ext.image_mm.round(1))))
edges = []
for i in keep:
    g = geoms[i]
    for k in ("apex", "base1", "base2"):
        r, p = swing.radial_profile(dishes[i]["img"][1], g[k])
        if p.max() > 1.5 * p[r < 0.3].mean():   # a clear rim of uptake around the footprint
            edges.append(swing.edge_radius(r, p))
edges = np.array(edges)
print(f"Radius of the dark footprint where the uptake rim is clear: median {np.nanmedian(edges):.2f} mm "
      f"(range {np.nanmin(edges):.2f} to {np.nanmax(edges):.2f}, {np.isfinite(edges).sum()} electrodes); needle outside radius 0.75 mm (Methods 4.2)")
print(f"Base midpoint to apex centre: {geo.midpoint_to_apex_mm.mean():.2f} mm; to the apex footprint edge: "
      f"{geo.midpoint_to_apex_mm.mean() - 0.75:.2f} mm; 32 ROIs of 200 um span {32 * 0.2:.1f} mm (Methods 4.4)")

Three checks are consistent with 0.65 um per pixel, the size the camera and objective of Methods
4.4 give. At that size the stitched images are 14.5 mm and 19.3 mm wide, the sizes of the glass
bottoms of 14 mm and 20 mm glass-bottom dishes; the dark footprints have the radius printed above,
a little less than the 0.75 mm outside radius of the needles; and 32 contiguous ROIs of 200 um
span about the distance from the base to the apex. At twice that pixel size none of these would
hold. The field strength along the bisector, a fourth check, is printed with the field models
below. The pixel size remains an assumption, as the files do not state it.

In [ ]:
grids = {i: swing.make_grid(geoms[i], dishes[i]["img"].shape) for i in keep}
volts_kv = {"1 Hz": 5.0, "833 kHz": 4.0}   # amplitudes in the session descriptions and file names
rows = []
for i in keep:
    g = geoms[i]
    for lay in ("L1", "L2"):
        aB, fB = swing.bisector_angles(g, lay, "B", None, grids[i])
        aC, fC = swing.bisector_angles(g, lay, "C", None)
        kv = volts_kv[info.freq[i]]
        rows.append({"subject": info.subject[i], "layout": lay, "angle_first_B": aB[0], "angle_last_B": aB[-1],
                     "angle_max_B": aB.max(), "angle_first_C": aC[0], "angle_last_C": aC[-1], "angle_max_C": aC.max(),
                     "E_kV_cm_min_B": (fB * kv * 10).min(), "E_kV_cm_max_B": (fB * kv * 10).max()})
ang = pd.DataFrame(rows)
print("Angle of field direction change and field strength (model B, one base electrode at the stated amplitude) along the ROIs:")
print(ang.round(1).to_string(index=False))
# Grid step check for model B on one dish
i0 = keep[0]
gfine = swing.GridModel(np.array([geoms[i0][k] for k in ("apex", "base1", "base2")]) * mpb,
                        np.array(dishes[i0]["img"].shape[-2:]) / 2 * mpb, h_mm=0.05)
a1, _ = swing.bisector_angles(geoms[i0], "L2", "B", None, grids[i0])
a2, _ = swing.bisector_angles(geoms[i0], "L2", "B", None, gfine)
print(f"\nModel B, grid 0.1 against 0.05 mm on {info.subject[i0]}: largest angle difference {np.abs(a1 - a2).max():.2f} degrees")

Along layout L2 the angle runs from 180 degrees beyond the base midpoint to near 0 next to the
apex, as Figure 1 describes; along L1, which starts at the base midpoint, it starts below 180
degrees, because the grounded apex pulls the field at the midpoint toward it. The field strength
along the ROIs is printed for each dish; Results 2.1 gives 5 to 10 kV/cm, which the values at
0.65 um per pixel are consistent with (at twice the pixel size the field would be about half).

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 13))
for ax, i in zip(axes.flat, keep):
    yp = dishes[i]["img"][1]
    n0 = yp.shape[0]
    lo, hi = np.percentile(yp[n0 // 4:3 * n0 // 4, n0 // 4:3 * n0 // 4], [1, 99.5])
    ax.imshow(yp, cmap="gray", vmin=lo, vmax=hi)
    g = geoms[i]
    for k, col in (("apex", "r"), ("base1", "c"), ("base2", "y")):
        ax.add_patch(plt.Circle(g[k][::-1], 0.75 / mpb, fill=False, color=col))
    for lay, col in (("L1", "m"), ("L2", "lime")):
        r = swing.roi_centres(g, lay)
        ax.plot(r[:, 1], r[:, 0], ".", color=col, ms=2)
    ax.set_title(f"{info.subject[i][7:]}\nYP {lo:.0f} to {hi:.0f} counts", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[len(keep):]:
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Display range: 1st to 99.5th percentile of the central half of each YP image (in titles); brighter pixels are clipped.")
print("Red: apex; cyan and yellow: base electrodes (circles of 1.5 mm); magenta: ROI layout L1; green: L2.")
for i in keep:
    print(f"{info.subject[i]}: YP block means {dishes[i]['img'][1].min():.0f} to {dishes[i]['img'][1].max():.0f}, "
          f"MemBrite {dishes[i]['img'][0].min():.0f} to {dishes[i]['img'][0].max():.0f} counts (0 and ~65535 are padding outside the tiles)")

In [ ]:
# Do the condition labels match the images? Uptake next to each electrode (ROI centred 0.95 mm from
# the centre toward the triangle's centroid), minus the footprint background, per dish.
rows = []
for i in keep:
    g = geoms[i]; yp = dishes[i]["img"][1]
    bg = swing.footprint_background(yp, g)
    v = {k: swing.square_mean(yp, swing.adjacent_rois(g, k, 0.0, "centroid")[0], np.array([1.0, 0.0])) - bg
         for k in ("apex", "base1", "base2")}
    rows.append({"subject": info.subject[i], "freq": info.freq[i], "train": info.train[i], **v,
                 "apex_over_max_base": v["apex"] / max(v["base1"], v["base2"])})
lab = pd.DataFrame(rows)
print(lab.round(2).to_string(index=False))
b13 = lab[lab.subject.str.contains("0,8MHz-BP-8-24-21-BPAE-13")]
print("\nRatio of apex to the larger base uptake, by group (min to max):")
print(lab.groupby(["freq", "train"]).apex_over_max_base.agg(["min", "max"]).round(2).to_string())

The images agree with their labels: in every paired dish the uptake next to the apex is larger
than next to either base electrode, and in every single-pulse dish the uptake next to one base
electrode is the largest. BPAE-13 of 8-24-21, whose session description is that of a two-electrode
bipolar experiment, shows the three footprints of the triangular array at the same spacing and
the paired-pulse pattern of the other two 833 kHz paired dishes; it is used as a paired dish, and
its description is recorded as a metadata error. The files have no time series, so there is no
clock to check against a trial table.

## Measures along the bisector

For every dish, layout and background reading, the mean YP intensity in each of the 32 ROIs of
200 x 200 um, the angle at each ROI under models B and C, and, as a variant, YP over MemBrite in
the same ROI.

- Background (claims file, as amended in reply to `questions/000631.yaml`): (b), which decides,
  is the ROI mean minus the mean YP within 0.5 mm of the three electrode centres of the same dish
  (Methods 4.4, "to account for 'spontaneous' YP uptake, measurements were also performed within
  the hollow electrodes' footprints"); (a), the ROI mean as measured, is a variant only.
- Layout (left open by the paper; see the helper): L1, 32 ROIs evenly spaced from the base
  midpoint to the apex footprint (the 32 do not fit there side by side, so they overlap by about
  20%); L2, 32 contiguous ROIs ending at the apex footprint and running back past the base
  midpoint.
- Angles: models B and C of the claims file, per paired dish, averaged over the paired dishes of a
  frequency. The controls use their own dish's ROIs at the same positions relative to the array.
- Controls: all unique single-pulse dishes of the same frequency; subsets shown as variants.

In [ ]:
rows = []
for i in keep:
    g = geoms[i]; img = dishes[i]["img"]
    bg = swing.footprint_background(img[1], g)
    for lay in ("L1", "L2"):
        yp = swing.bisector_profile(img[1], g, lay)
        mb = swing.bisector_profile(img[0], g, lay)
        aB, _ = swing.bisector_angles(g, lay, "B", None, grids[i])
        aC, _ = swing.bisector_angles(g, lay, "C", None)
        for k in range(swing.N_ROI):
            rows.append({"subject": info.subject[i], "freq": info.freq[i], "train": info.train[i], "date": info.date[i],
                         "layout": lay, "roi": k, "yp": yp[k], "membrite": mb[k], "bg": bg, "angle_B": aB[k], "angle_C": aC[k]})
prof = pd.DataFrame(rows)
print("Footprint background (counts) per dish:")
print(prof.groupby(["freq", "train", "subject"]).bg.first().round(1).to_string())
print("\nROI means (YP counts), range over all dishes and ROIs:", prof.yp.min().round(1), "to", prof.yp.max().round(1))
print("ROIs whose background-subtracted value is 0 or below:", int(((prof.yp - prof.bg) <= 0).sum()), "of", len(prof))

def curves(freq, layout, model, bg, norm, measure="yp", controls=None, paired=None):
    """Returns angle per ROI (mean over paired dishes), the normalised values (rows: per-pair
    ratios for (i), one row for (ii)), and the paired and single matrices."""
    v = prof[(prof.freq == freq) & (prof.layout == layout)].copy()
    if measure == "yp":
        v["val"] = v.yp - (v.bg if bg == "b" else 0)
    else:   # YP over MemBrite, background subtracted from YP only
        v["val"] = (v.yp - (v.bg if bg == "b" else 0)) / v.membrite
    P = v[v.train == "paired"].pivot(index="subject", columns="roi", values="val")
    S = v[v.train == "single"].pivot(index="subject", columns="roi", values="val")
    if controls is not None:
        S = S.loc[[s for s in S.index if controls(s)]]
    if paired is not None:
        P = P.loc[[s for s in P.index if paired(s)]]
    angles = v[(v.train == "paired") & v.subject.isin(P.index)].groupby("roi")[f"angle_{model}"].mean().values
    return angles, swing.normalised(P.values, S.values, norm), P, S

READINGS = [(lay, model, bg, norm) for lay in ("L1", "L2") for model in ("B", "C") for bg in ("a", "b") for norm in ("i", "ii")]
print("\nReadings run for each Figure 2 measure:", len(READINGS), "(layout x angle model x background x normalisation)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for ax, (lay, bg) in zip(axes.flat, [(l, b) for b in ("a", "b") for l in ("L1", "L2")]):
    for freq, col in (("1 Hz", "tab:blue"), ("833 kHz", "tab:red")):
        a, r, P, S = curves(freq, lay, "B", bg, "ii")
        # s.e. over paired dishes, each divided by the mean control
        se = (100 * P.values / S.values.mean(0)).std(0, ddof=1) / np.sqrt(len(P))
        ax.errorbar(a, r[0], yerr=se, fmt="o-", ms=3, color=col, label=f"{freq} (paired n={len(P)}, controls n={len(S)})")
        print(f"{lay} background {bg} {freq}: plotted values {r[0].min():.1f} to {r[0].max():.1f} %, angles {a.min():.0f} to {a.max():.0f} deg")
    ax.axhline(100, color="k", lw=0.8)
    ax.set_title(f"layout {lay}, background {bg}, normalisation (ii), angles model B")
    ax.set_xlabel("angle of field direction change (deg)"); ax.set_ylabel("paired YP, % of single")
    ax.invert_xaxis(); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("All points are within the axes.")
for bg in ("a", "b"):
    tab = {}
    for freq in ("1 Hz", "833 kHz"):
        a, r, P, S = curves(freq, "L2", "B", bg, "ii")
        tab[f"angle {freq}"] = a; tab[f"% {freq}"] = r[0]
    print(f"\nLayout L2, model B, background {bg}, normalisation (ii), ROI 0 (base end) to 31 (apex end):")
    print(pd.DataFrame(tab).round(1).T.to_string())

In [ ]:
def combine(outcomes, bg="b"):
    """Outcome over the angle models, per layout, under one background reading. Reading (b)
    decides (claims file, opening comment, as amended in reply to questions/000631.yaml: the
    paper's values are corrected for spontaneous uptake, and (a) is a variant only); called with
    bg="a" it gives the variant. Blocked as methods_underspecified when angle models B and C give
    different outcomes."""
    out = {}
    for lay in ("L1", "L2"):
        per_model = {model: outcomes[(lay, model, bg)] for model in ("B", "C")}
        out[lay] = per_model["B"] if per_model["B"] == per_model["C"] else "blocked (angle models differ)"
    return out


## fig2-180-cancellation

> At a 180 degree change of field direction (near the base electrodes), paired-pulse trains give
> less YP uptake than single-pulse trains, 2-fold less at 1 Hz and 20-fold less at 833 kHz.

Criterion (claims file): per ROI, paired as % of single, under normalisations (i) and (ii), the
fold being 100 / %. "At 180 degrees" is (1) the ROI of largest angle or (2) the ROI of lowest
mean value; a value holds (2 +- 0.2 at 1 Hz, 20 +- 2 at 833 kHz) under any of these with either
normalisation. Reproduced when both hold, under background reading (b); reading (a) is a variant
only.

In [ ]:
EXPECT_180 = {"1 Hz": (2.0, 0.2), "833 kHz": (20.0, 2.0)}
rows = []
for lay, model, bg, norm in READINGS:
    for freq in ("1 Hz", "833 kHz"):
        a, r, P, S = curves(freq, lay, model, bg, norm)
        c = r.mean(0)
        for rd in (1, 2):
            val, k = swing.at_180(c, a, rd)
            fold = 100 / val if val > 0 else np.inf
            e, t = EXPECT_180[freq]
            rows.append({"layout": lay, "model": model, "bg": bg, "norm": norm, "freq": freq, "reading": rd,
                         "roi": k, "angle": a[k], "pct": val, "fold": fold, "holds": abs(fold - e) <= t})
t180 = pd.DataFrame(rows)
print(t180.pivot_table(index=["layout", "model", "bg", "reading"], columns=["freq", "norm"], values="fold").round(2).to_string())
hold = t180.groupby(["layout", "model", "bg", "freq"]).holds.any().unstack("freq")
hold["both hold"] = hold.all(axis=1)
print("\nDoes each value hold under some normalisation and 180-degree reading?")
print(hold.to_string())
res180 = {k: ("criterion met" if v else "criterion not met") for k, v in hold["both hold"].items()}
out180 = combine(res180)
print("\nOutcome per layout, background (b), which decides:", out180)
print("Variant, background (a), deciding nothing:", combine(res180, "a"))

In [ ]:
# Variants: control subsets, the 833 kHz paired dish with the wrong description left out, YP over MemBrite.
variants = {
    "all controls": dict(),
    "1 Hz controls of 7-31-21 only": dict(controls=lambda s: "7-31-21" in s),
    "1 Hz controls of 8-9-21 only (day of the paired dishes)": dict(controls=lambda s: "8-9-21" in s),
    "833 kHz without BPAE-13 of 8-24-21": dict(paired=lambda s: not s.endswith("8-24-21-BPAE-13")),
    "YP / MemBrite": dict(measure="ratio"),
}
rows = []
for name, kw in variants.items():
    for bg in ("a", "b"):
        for freq in ("1 Hz", "833 kHz"):
            if ("1 Hz" in name and freq != "1 Hz") or ("833" in name and freq != "833 kHz"):
                continue
            for norm in ("i", "ii"):
                a, r, P, S = curves(freq, "L2", "B", bg, norm, **kw)
                c = r.mean(0)
                v1, _ = swing.at_180(c, a, 1); v2, _ = swing.at_180(c, a, 2)
                rows.append({"variant": name, "bg": bg, "freq": freq, "norm": norm, "n_paired": len(P), "n_single": len(S),
                             "fold_reading1": 100 / v1 if v1 > 0 else np.inf, "fold_reading2": 100 / v2 if v2 > 0 else np.inf})
print("Layout L2, model B:")
print(pd.DataFrame(rows).round(2).to_string(index=False))
# per paired dish, fold at the ROI of largest angle against the mean control (bg b, L2)
rows = []
for freq in ("1 Hz", "833 kHz"):
    a, r, P, S = curves(freq, "L2", "B", "b", "ii")
    k = int(np.argmax(a))
    for s in P.index:
        rows.append({"freq": freq, "paired dish": s, "pct_at_largest_angle": 100 * P.loc[s].values[k] / S.values.mean(0)[k]})
print("\nPer paired dish (L2, model B, background b), % of the mean control at the ROI of largest angle:")
print(pd.DataFrame(rows).round(1).to_string(index=False))

**Result: partially reproduced.** Paired trains give less uptake than single trains at the
largest angles at both frequencies under every reading, and less at 833 kHz than at 1 Hz, but the
two fold values never both fall within tolerance. With the footprint background subtracted
(reading b), layout L2 gives 2.42 to 2.51 fold at 1 Hz (expected 2 +- 0.2) and 11.77 to 21.98 fold
at 833 kHz (the ROI of lowest value, 18.52 to 21.98, is within 20 +- 2); layout L1 gives 1.92 to
2.48 fold at 1 Hz (1.92 and 2.19 within tolerance) and 11.22 to 11.48 fold at 833 kHz. Without subtraction
(reading a) the values are 1.26 to 1.55 and 2.66 to 3.38 fold, neither within tolerance; this
variant decides nothing under the amended claims file, and gives the same outcome. The two angle
models give the same values. The cancellation is present, and its size matches at one frequency at
a time. Per
paired dish at 833 kHz, the ROI of largest angle is 5.8 to 9.0% of the mean control. The outcome
rests on the assumed pixel size and ROI layout.

## fig2-crossover-angle

> Cancellation weakens as the angle is reduced, and paired and single pulses become equally
> effective at about 160 degrees at 1 Hz and about 130 degrees at 833 kHz.

Criterion: the largest angle at which the mean paired value reaches 100%, by linear interpolation
between neighbouring ROIs; both angles within tolerance (160 +- 16, 130 +- 13) under one
normalisation at least, under background reading (b); reading (a) is a variant only.

In [ ]:
EXPECT_X = {"1 Hz": (160.0, 16.0), "833 kHz": (130.0, 13.0)}
rows = []
for lay, model, bg, norm in READINGS:
    for freq in ("1 Hz", "833 kHz"):
        a, r, P, S = curves(freq, lay, model, bg, norm)
        x = swing.crossover(r.mean(0), a)
        e, t = EXPECT_X[freq]
        rows.append({"layout": lay, "model": model, "bg": bg, "norm": norm, "freq": freq, "crossover": x,
                     "holds": bool(abs(x - e) <= t)})
tx = pd.DataFrame(rows)
print(tx.pivot_table(index=["layout", "model", "bg"], columns=["freq", "norm"], values="crossover").round(1).to_string())
both = tx.groupby(["layout", "model", "bg", "norm"]).holds.all()
dec = both.groupby(["layout", "model", "bg"]).any()
print("\nBoth angles within tolerance under one normalisation:")
print(dec.to_string())
resx = {k: ("criterion met" if v else "criterion not met") for k, v in dec.items()}
outx = combine(resx)
print("\nOutcome per layout, background (b), which decides:", outx)
print("Variant, background (a), deciding nothing:", combine(resx, "a"))
# Variants on L2, model B
rows = []
for name, kw in variants.items():
    for bg in ("a", "b"):
        for freq in ("1 Hz", "833 kHz"):
            if ("1 Hz" in name and freq != "1 Hz") or ("833" in name and freq != "833 kHz"):
                continue
            for norm in ("i", "ii"):
                a, r, P, S = curves(freq, "L2", "B", bg, norm, **kw)
                rows.append({"variant": name, "bg": bg, "freq": freq, "norm": norm, "crossover": swing.crossover(r.mean(0), a)})
print("\nVariants, layout L2, model B:")
print(pd.DataFrame(rows).pivot_table(index=["variant", "freq"], columns=["bg", "norm"], values="crossover").round(1).to_string())

**Result: reproduced.** The mean paired value reaches 100% at 145.3 to 149.6 degrees at 1 Hz
(expected 160 +- 16) and at 124.1 to 125.9 degrees at 833 kHz (expected 130 +- 13), under all 16
readings (layout, angle model, background, normalisation), so the 8 readings under background
(b), which decide, and the variant (a) agree. The 1 Hz angle is 1.3 to 5.6 degrees
above the lower bound of its tolerance; with only the two controls of 8-9-21 it is 130.5 to 161.4
degrees, and with only those of 7-31-21, 149.3 to 149.8. In the curve tables above, every ROI above the
crossover angle is below 100% and the ROIs next to the crossover are higher than those at the
largest angles, so cancellation weakens as the angle falls; below the crossover most ROIs are above
100%, with a dip at 1 Hz between 132 and 143 degrees (79 to 97%, normalisation ii). The
outcome rests on the assumed pixel size.

## fig2-facilitation-plateau

> Below the crossover, paired pulses enhance electroporation up to about three-fold over single
> pulses at 90 to 100 degrees for both frequencies, followed by a plateau down to 0 degrees, so
> pulses at 90 degrees and at smaller angles are as efficient as co-directional pulses.

Criterion: level (1) the mean of the ROIs between 90 and 100 degrees, or (2) the mean of the ROIs
from 100 degrees to the smallest angle, 3 +- 0.3 fold, either normalisation. Plateau readings over
the ROIs from 0 to 90 degrees: (P1) a linear fit whose slope's 95% CI includes zero; (P2) the mean
of the ROIs below 10 degrees within 10% of the mean of the ROIs between 90 and 100 degrees. The
angle where the plateau is reached is the largest angle at which the value first comes within
10% of the level, and must lie within 81 to 110 degrees. All at both frequencies, under
background reading (b); blocked when the plateau readings disagree. Reading (a) is a variant only.

In [ ]:
def plateau_measures(c, a):
    lv1 = c[(a >= 90) & (a <= 100)]
    lv1 = lv1.mean() / 100 if len(lv1) else np.nan
    lv2 = c[a <= 100].mean() / 100
    sel = a <= 90
    fit = stats.linregress(a[sel], c[sel])
    tcrit = stats.t.ppf(0.975, sel.sum() - 2)
    p1 = bool(abs(fit.slope) <= tcrit * fit.stderr)
    low = c[a < 10]
    m90 = c[(a >= 90) & (a <= 100)]
    p2 = bool(len(low) and len(m90) and abs(low.mean() - m90.mean()) <= 0.1 * m90.mean())
    o = np.argsort(a)[::-1]
    def reach(level):
        if not np.isfinite(level):
            return np.nan
        hit = [a[j] for j in o if abs(c[j] / 100 - level) <= 0.1 * level]
        return hit[0] if hit else np.nan
    return {"level1": lv1, "level2": lv2, "angle_at_level1": reach(lv1), "angle_at_level2": reach(lv2),
            "slope_pct_per_deg": fit.slope, "slope_ci95": tcrit * fit.stderr, "n_le90": int(sel.sum()), "P1": p1,
            "n_below10": int(len(low)), "P2": p2}

rows = []
for lay, model, bg, norm in READINGS:
    for freq in ("1 Hz", "833 kHz"):
        a, r, P, S = curves(freq, lay, model, bg, norm)
        rows.append({"layout": lay, "model": model, "bg": bg, "norm": norm, "freq": freq, **plateau_measures(r.mean(0), a)})
tp = pd.DataFrame(rows)
print(tp.round(2).to_string(index=False))

def ok(row, plateau):
    good = []
    for lv in ("1", "2"):
        level, angle = row[f"level{lv}"], row[f"angle_at_level{lv}"]
        good.append(bool(abs(level - 3) <= 0.3 and 81 <= angle <= 110 and row[plateau]))
    return any(good)

rows = []
for (lay, model, bg), grp in tp.groupby(["layout", "model", "bg"]):
    for plateau in ("P1", "P2"):
        per_freq = {f: any(ok(r, plateau) for _, r in grp[grp.freq == f].iterrows()) for f in ("1 Hz", "833 kHz")}
        rows.append({"layout": lay, "model": model, "bg": bg, "plateau": plateau, **per_freq, "both hold": all(per_freq.values())})
dp = pd.DataFrame(rows)
print("\nLevel, angle and plateau all hold (some normalisation and level reading), per frequency:")
print(dp.to_string(index=False))
outp = {}
for plateau in ("P1", "P2"):
    resp = {(r["layout"], r["model"], r["bg"]): ("criterion met" if r["both hold"] else "criterion not met")
            for _, r in dp[dp.plateau == plateau].iterrows()}
    outp[plateau] = combine(resp)
    print(f"Variant, background (a), plateau reading {plateau}, deciding nothing:", combine(resp, "a"))
print("\nOutcome per plateau reading and layout, background (b), which decides:", outp)

**Result: partially reproduced.** Facilitation below the crossover is present at both
frequencies, but the three-fold plateau from 90 to 100 degrees holds only at 833 kHz. At 833 kHz
with background (b), the level at 90 to 100 degrees is 2.29 to 2.78 fold and the mean from 100
degrees down is 2.80 to 2.96 fold; on layout L2 the level is reached at 94.6 to 96.1 degrees and
the slope over 0 to 90 degrees has a 95% CI that includes zero (P1). At 1 Hz there is no plateau:
the value keeps rising as the angle falls (slope -1.35 to -3.41 % per degree over 0 to 90
degrees, CI excluding zero under every reading), the level at 90 to 100 degrees is 1.15 to 1.36 fold, and the mean from 100 degrees down 2.07 to
2.36 fold. In the variant without background subtraction (a), which decides nothing, no level is within 2.7
to 3.3 (1.08 to 1.74 fold).
Plateau reading P2 has no ROI below 10 degrees under angle model C, and is met only at 833 kHz on
L2 with model B and normalisation (ii). Under background (b) the criterion is not met under either
plateau reading, so the claim is not blocked.

## fig2-frequency-dependence

> Pairing pulses enhances electroporation at small angles and inhibits it at large angles in a
> frequency-dependent manner, with stronger cancellation at 833 kHz than at 1 Hz.

Criterion: under one normalisation at least, the value at the 180 degree reading is lower at
833 kHz than at 1 Hz, the 833 kHz crossover angle is smaller than the 1 Hz one, and at both
frequencies the value is above 100% at the smallest angle and below 100% at the largest; under
background reading (b), with (a) a variant only.

In [ ]:
rows = []
for lay, model, bg, norm in READINGS:
    d = {}
    for freq in ("1 Hz", "833 kHz"):
        a, r, P, S = curves(freq, lay, model, bg, norm)
        c = r.mean(0)
        d[freq] = {"v180_1": swing.at_180(c, a, 1)[0], "v180_2": swing.at_180(c, a, 2)[0], "x": swing.crossover(c, a),
                   "small": c[np.argmin(a)], "large": c[np.argmax(a)]}
    lower = any(d["833 kHz"][k] < d["1 Hz"][k] for k in ("v180_1", "v180_2"))
    xs = d["833 kHz"]["x"] < d["1 Hz"]["x"]
    ends = all(d[f]["small"] > 100 and d[f]["large"] < 100 for f in d)
    rows.append({"layout": lay, "model": model, "bg": bg, "norm": norm,
                 "v180_1Hz": d["1 Hz"]["v180_1"], "v180_833": d["833 kHz"]["v180_1"],
                 "min_1Hz": d["1 Hz"]["v180_2"], "min_833": d["833 kHz"]["v180_2"],
                 "x_1Hz": d["1 Hz"]["x"], "x_833": d["833 kHz"]["x"],
                 "smallest_angle_1Hz": d["1 Hz"]["small"], "smallest_angle_833": d["833 kHz"]["small"],
                 "lower_at_833": lower, "crossover_smaller_at_833": xs, "ends_ok": ends, "all": lower and xs and ends})
tf = pd.DataFrame(rows)
print(tf.round(1).to_string(index=False))
decf = tf.groupby(["layout", "model", "bg"])["all"].any()
resf = {k: ("criterion met" if v else "criterion not met") for k, v in decf.items()}
outf = combine(resf)
print("\nOutcome per layout, background (b), which decides:", outf)
print("Variant, background (a), deciding nothing:", combine(resf, "a"))

**Result: reproduced.** Under all 16 readings, the value at the 180 degree reading is lower at
833 kHz than at 1 Hz (on L2 with background b, 8.5 against 41.3% at the ROI of largest angle;
with background a, 37.6 against 79.3%), the crossover angle is smaller at 833 kHz (124.1 to 125.9
against 145.3 to 149.6 degrees), and at both frequencies the value is above 100% at the smallest
angle (195.9 to 323.4%) and below 100% at the largest (7.9 to 79.3%). The outcome is decided by
the 8 readings under background (b); the variant (a) agrees.

## fig2-60-fold-co-counter

> At 833 kHz, co-directional and counter-directional pulse pairs differ about 60-fold in
> electroporation efficiency.

Criterion: at 833 kHz, (a) the ROI of smallest angle over the ROI of largest angle, or (b) the
plateau level of fig2-facilitation-plateau over the 180 degree value of fig2-180-cancellation;
60 +- 6 under one of these and one normalisation, under background reading (b); reading (a) is a
variant only.

In [ ]:
rows = []
for lay, model, bg, norm in READINGS:
    a, r, P, S = curves("833 kHz", lay, model, bg, norm)
    c = r.mean(0)
    pm = plateau_measures(c, a)
    v1, v2 = swing.at_180(c, a, 1)[0], swing.at_180(c, a, 2)[0]
    ratio = lambda x, y: x / y if y > 0 else np.inf
    cand = {"a_ends": ratio(c[np.argmin(a)], c[np.argmax(a)]),
            "b_level1_over_180r1": ratio(100 * pm["level1"], v1), "b_level1_over_180r2": ratio(100 * pm["level1"], v2),
            "b_level2_over_180r1": ratio(100 * pm["level2"], v1), "b_level2_over_180r2": ratio(100 * pm["level2"], v2)}
    # uptake in the paired dishes alone (not normalised), smallest over largest angle
    praw = P.values.mean(0)
    rows.append({"layout": lay, "model": model, "bg": bg, "norm": norm, **cand,
                 "paired_raw_ends": ratio(praw[np.argmin(a)], praw[np.argmax(a)]),
                 "holds": any(abs(v - 60) <= 6 for v in cand.values())})
t60 = pd.DataFrame(rows)
print(t60.round(1).to_string(index=False))
dec60 = t60.groupby(["layout", "model", "bg"]).holds.any()
res60 = {k: ("criterion met" if v else "criterion not met") for k, v in dec60.items()}
out60 = combine(res60)
print("\nOutcome per layout, background (b), which decides:", out60)
print("Variant, background (a), deciding nothing:", combine(res60, "a"))
cand_cols = ["a_ends", "b_level1_over_180r1", "b_level1_over_180r2", "b_level2_over_180r1", "b_level2_over_180r2"]
within = t60[t60.bg == "b"].set_index(["layout", "model", "norm"])[cand_cols].apply(lambda s: (s - 60).abs() <= 6)
print("\nBackground (b): which ratios are within 60 +- 6, per layout, model and normalisation:")
print(within.to_string())

**Result: reproduced, on layout L2, which is adopted as an assumption.** Under background (b),
which decides, layout L2 gives a plateau level over the ROI of lowest value of 51.5 to 61.6 fold
under both angle models, three of its four values within 60 +- 6 (level reading 1 with
normalisation ii, 56.8; level reading 2, 54.8 and 61.6); the other ratios are 32.7 to 36.2 fold.
On layout L1 every ratio under (b) is 25.8 to 34.5 fold, so the criterion is not met there; L2 is
adopted where the layouts differ, as an assumption (see the summary). The variant without
background subtraction (a), which decides nothing under the amended claims file, gives 3.6 to 5.8
fold on both layouts. The claim was blocked before the reader's answer only because (a) and (b)
disagreed. Paired uptake alone, not normalised to the controls, is 25.0 times higher at the apex
end than at the base end on L2 with background (b), and 4.5 times with (a).


## fig4-unipolar-pattern

> With single-pulse trains on one base electrode, YP uptake follows the electric field map; with
> paired pulses on both base electrodes at 833 kHz, uptake is strong near the grounded apex
> electrode and weak or absent near the base electrodes despite the strong field there.

Criterion: mean YP in 200 x 200 um ROIs adjacent to the apex and to each base electrode, under
background reading (b), with (a) a variant only; in the paired sample the apex exceeds each base, and in the single
sample the energised base exceeds the apex. "Near": ROIs touching the footprint edge (1.5 mm
diameter) and ROIs 200 um farther out, both run. The match to the field map is the Spearman
correlation of ROI uptake with field strength, with no threshold. The 833 kHz, 4 kV dishes are
the dishes of Figure 4 (Results 2.3). Which side of the electrode the ROI is on is not given; the
ROI on the line to the triangle's centroid and the mean of eight ROIs around the electrode are
both run. The files do not say which base electrode was energised in the single-pulse dishes;
the base with the larger adjacent uptake is taken as energised, and the other is shown.

In [ ]:
rows = []
for i in keep:
    if info.freq[i] != "833 kHz":
        continue
    g = geoms[i]; yp = dishes[i]["img"][1]
    bgv = swing.footprint_background(yp, g)
    for gap in (0.0, 0.2):
        for direction in ("centroid", "ring"):
            v = {k: np.mean([swing.square_mean(yp, c, np.array([1.0, 0.0])) for c in swing.adjacent_rois(g, k, gap, direction)])
                 for k in ("apex", "base1", "base2")}
            for bg in ("a", "b"):
                sub = bgv if bg == "b" else 0.0
                rows.append({"subject": info.subject[i], "train": info.train[i], "gap_mm": gap, "side": direction, "bg": bg,
                             **{k: v[k] - sub for k in v}})
t4 = pd.DataFrame(rows)
t4["base_hi"] = t4[["base1", "base2"]].max(axis=1)
t4["base_lo"] = t4[["base1", "base2"]].min(axis=1)
print("Per dish:")
print(t4.round(1).to_string(index=False))
m4 = t4.groupby(["train", "gap_mm", "side", "bg"])[["apex", "base1", "base2", "base_hi", "base_lo"]].mean()
print("\nMean over dishes:")
print(m4.round(1).to_string())
rows = []
for (gap, side, bg), grp in t4.groupby(["gap_mm", "side", "bg"]):
    pr = grp[grp.train == "paired"]; sg = grp[grp.train == "single"]
    paired_ok = bool(pr.apex.mean() > pr.base1.mean() and pr.apex.mean() > pr.base2.mean())
    # energised base per dish = the base with larger uptake; compare its mean with the apex mean
    single_ok = bool(sg.base_hi.mean() > sg.apex.mean())
    rows.append({"gap_mm": gap, "side": side, "bg": bg,
                 "paired apex/base (each dish, min)": (pr.apex / pr.base_hi).min(),
                 "paired dishes apex > both bases": int((pr.apex > pr.base_hi).sum()),
                 "single energised/apex (each dish, min)": (sg.base_hi / sg.apex).min(),
                 "single dishes energised > apex": int((sg.base_hi > sg.apex).sum()),
                 "single other base/apex (mean)": sg.base_lo.mean() / sg.apex.mean(),
                 "paired_ok": paired_ok, "single_ok": single_ok, "both": paired_ok and single_ok})
d4 = pd.DataFrame(rows)
print("\nPer reading (means over 3 paired and 3 single dishes):")
print(d4.round(2).to_string(index=False))
print("\nAll readings agree:", d4.both.nunique() == 1, "; outcome under every reading:", bool(d4.both.all()))
print("Background (b), which decides: criterion met under every reading:", bool(d4[d4.bg == "b"].both.all()),
      "; variant, background (a):", bool(d4[d4.bg == "a"].both.all()))

In [ ]:
# Match to the field map: Spearman correlation over a grid of 200 um ROIs within 4 mm of the
# triangle's centroid, outside the footprints, background b. Single dishes: field of the energised
# base (the base with larger adjacent uptake); paired dishes: the same field of one base, shown as a contrast.
rows = []
for i in keep:
    if info.freq[i] != "833 kHz":
        continue
    g = geoms[i]; yp = dishes[i]["img"][1]
    pts = swing.tile_rois(g)
    up = np.array([swing.square_mean(yp, p, np.array([1.0, 0.0])) for p in pts]) - swing.footprint_background(yp, g)
    sub = t4[(t4.subject == info.subject[i]) & (t4.gap_mm == 0.0) & (t4.side == "centroid") & (t4.bg == "b")].iloc[0]
    energised = "base1" if sub.base1 >= sub.base2 else "base2"
    for model in ("B", "C"):
        E = swing.field_at(g, pts, energised, model, grids[i])
        rho, p = stats.spearmanr(up, E)
        rows.append({"subject": info.subject[i], "train": info.train[i], "energised (taken)": energised, "model": model,
                     "n_rois": len(pts), "spearman_rho": rho, "p": p})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** In each of the three 833 kHz paired dishes, uptake next to the apex
exceeds uptake next to both base electrodes under all eight readings (ROI touching the footprint
or 200 um out, on the line to the centroid or around the electrode, background b, which decides,
or the variant a); the
smallest per-dish ratio of apex to the larger base is 1.86 to 3.68 across readings. In each of the
three single-pulse dishes, uptake next to the base taken as energised exceeds uptake next to the
apex (smallest per-dish ratio 1.40 to 2.18); the other base also exceeds the apex on average
(1.25 to 2.31). Over the dish, uptake in the single-pulse dishes follows the field strength of the
energised base (Spearman rho 0.50 to 0.96, both field models), while in the paired dishes it does
not (rho -0.10 to -0.26). Which base was energised is taken from the uptake, as the files do not
say.

## Summary

| Claim | Outcome | Rests on an assumption |
|---|---|---|
| fig2-180-cancellation | partially reproduced | pixel size, ROI layout |
| fig2-crossover-angle | reproduced | pixel size, ROI layout |
| fig2-facilitation-plateau | partially reproduced | pixel size, ROI layout |
| fig2-frequency-dependence | reproduced | pixel size, ROI layout |
| fig2-60-fold-co-counter | reproduced | pixel size, ROI layout |
| fig3-1hz-co-equals-cross | blocked (data_missing) | |
| fig3-770khz-co-enhanced | blocked (data_missing) | |
| fig3-770khz-cross-same-as-1hz | blocked (data_missing) | |
| fig3-no-area-dependence | blocked (data_missing) | |
| fig4-unipolar-pattern | reproduced | pixel size |
| fig5-base-sawtooth | blocked (data_missing) | |
| fig5-apex-larger-polarization | blocked (data_missing) | |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.**

- The paper measured 32 ROIs in MetaMorph; the files hold no ROIs, so they are placed here from
  the electrode footprints found in each image. Their layout along the dashed line of Figure 7B
  is not given. Two layouts were run. L2 (32 contiguous ROIs from the apex footprint back past the
  base midpoint) is adopted where the two disagree (fig2-60-fold-co-counter), as an assumption:
  it is the one whose angles run from 180 degrees at the base to near 0 at the apex, as Results
  2.1 and Figure 1 describe, and it keeps the ROIs at 200 um side by side; L1 cannot reach
  180 degrees, and its 32 ROIs overlap.
- The angle of each ROI is computed from two field models (claims file readings B and C), as the
  paper gives neither the angles nor how they were obtained. Both give the same outcomes.
- The pixel size is taken as 0.65 um, from the camera and objective of Methods 4.4, and is
  consistent with the needle footprints, the base-to-apex distance and the field strength the
  paper gives; the files do not state it.
- Controls: all five unique 1 Hz single-pulse dishes (from two days) and the three 833 kHz ones;
  the paper does not say how many control dishes were used or how they were paired with the
  paired dishes. Subsets are shown as variants; the 1 Hz crossover moves with them.
- Spontaneous uptake is corrected by subtracting the mean YP in the electrode footprints of the
  same dish (background reading b). The paper says the footprints were measured "to account for"
  it (Methods 4.4) but not how. A reader, asked whether the uncorrected reading (a) is one the
  paper allows, answered that it is not, and the claims file now decides every claim under (b).
  Reading (a) is still shown as a variant: the ROI means include about 200 to 270 counts of camera
  offset and spontaneous uptake (the footprint values), so it cannot give a 20-fold cancellation.
  The answer changed one outcome, fig2-60-fold-co-counter, from blocked to reproduced.

**What was found about the dataset.**

- BPAE-03 and BPAE-18 of 7-31-21 (1 Hz, single pulses) are the same recording deposited twice
  under two subjects (identical image chunks and start time).
- BPAE-13 of 8-24-21, filed as an 833 kHz paired-pulse dish, carries the session description and
  id of a different experiment (bipolar pulses between two parallel electrodes); its image shows
  the triangular array and the paired-pulse pattern.
- The files hold no pixel size (resolution -1, unit "n.a."), no electrode positions, no ROIs, no
  angles and no measured values; which base electrode was energised in the single-pulse dishes is
  not stated.
- The ImageSeries description ("144 images captured in a 12x12 grid from a six-well plate")
  does not fit the 35 mm dishes and 216 images of the paper.
- The data of Figures 3 and 5 are not in the dandiset.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")